# Stocker — End-to-end Colab workflow

Self-contained: clones the repo, installs deps, builds the dataset, loads
**`google/gemma-4-E4B-it`** in-process via `transformers` (4-bit BnB),
pre-caches all 7 specialist votes, runs **GRPO** on the moderator LoRA via
TRL, and saves loss / reward plots + the trained adapter.

Designed for a free **Colab T4** (16 GB VRAM) — no separate vLLM server
needed. If you have an L4/A100, drop `load_in_4bit=False` for full bf16.

Outputs live under `training/runs/<timestamp>/`:
- `moderator-lora/` — trained PEFT adapter
- `loss.png`, `reward.png` — training curves
- `eval_pre.json`, `eval_post.json` — pre/post backtest reports

> Tip: `Runtime → Change runtime type → T4 GPU` before running.

In [ ]:
# Install — keep Colab's pre-baked pandas/numpy (TF + cudf depend on them);
# only upgrade what we strictly need.
#
# Colab pre-installs torchao==0.10 which PEFT ≥0.13 rejects (needs ≥0.16),
# so we upgrade it here too.
!pip install -q -U 'transformers>=4.55' 'trl>=0.11' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.43' 'datasets>=3.0' 'torchao>=0.16'
!pip install -q --upgrade-strategy=only-if-needed yfinance mplfinance pyarrow 'pydantic>=2' pydantic-settings 'openai>=1' tensorboard 'huggingface_hub>=1.10'
# Indicators are computed in-repo (app/data/indicators.py) — no pandas-ta.

In [ ]:
import os, sys, pathlib

# 1) Already in a stocker repo? (running from VSCode on a local clone, or
#    the repo was manually uploaded to Colab). Detect and skip the clone.
def _is_stocker_root(p: str) -> bool:
    return os.path.isfile(os.path.join(p, "app", "council", "specialists.py"))

CANDIDATES = [os.getcwd(), "/content/stocker", "/workspace/stocker"]
WORKDIR = next((c for c in CANDIDATES if _is_stocker_root(c)), None)

if WORKDIR is None:
    # 2) Not present — clone. EDIT THIS URL before running on a fresh Colab.
    REPO_URL = "https://github.com/CRIMSONHydra/stocker.git"
    WORKDIR  = "/content/stocker"
    assert "<your-username>" not in REPO_URL, (
        "Edit REPO_URL in this cell to your fork before running on a fresh runtime."
    )
    !git clone {REPO_URL} {WORKDIR}
    assert _is_stocker_root(WORKDIR), f"Clone failed — {WORKDIR}/app/council/ missing."

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("Working dir:", WORKDIR)

Working dir: /content/stocker


In [3]:
# Gemma 4 is Apache-2.0 (not gated) — token just lifts download rate limits.
from huggingface_hub import login
import getpass
login(token=getpass.getpass("HF token (read scope is enough): "))

In [ ]:
# Build the bundled dataset (yfinance + indicators + chart PNGs).
# Idempotent — skip if already done.
!python scripts/build_dataset.py
!python scripts/validate_tasks.py

[1/6] Fetching OHLCV ...
  prices: 536 rows
[2/6] Computing indicators ...
  indicators: 536 rows
[3/6] Fetching peers + commodity ...
  peers: 374 rows
[4/6] Loading curated news/forums/macro ...
  news: 25 headlines, forums: 17 posts, macro: 12 events
[5/6] Rendering candlestick charts ...


In [16]:
# Configure council client → HF Inference Endpoint (llama.cpp, Gemma 4 26B A4B Q4_K_M)
import os, getpass

# ── Your endpoint details (from HF dashboard) ───────────────────────────────
ENDPOINT_URL = "https://at0e6z2u64774tc7.us-east-1.aws.endpoints.huggingface.cloud/v1"
MODEL_NAME   = "ggml-org/gemma-4-26B-A4B-it-GGUF"  # llama.cpp ignores model name but keep it readable
# ────────────────────────────────────────────────────────────────────────────

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")

os.environ["API_BASE_URL"] = ENDPOINT_URL
os.environ["MODEL_NAME"]   = MODEL_NAME

from app.council.llm import build_openai_client_from_env
client = build_openai_client_from_env()
print(f"Endpoint → {client.base_url}")
print(f"Model    → {client.model}")

# Sanity ping — should reply in <3 s when endpoint is warm
ping = client.complete(
    [{"role": "user", "content": "Reply with the single word OK."}],
    max_tokens=5, temperature=0.0,
)
print(f"Ping reply: {ping!r}")


In [ ]:
# Pre-cache the 7 specialists' votes for task_easy. Specialists are
# FROZEN, so this runs once and is reused across every GRPO step.
# Resumable — already-cached items are skipped in milliseconds.
import sys, time
from datetime import datetime
from tqdm import tqdm

from app.council.runner import Council
from app.core.environment import StockerEnv

TRAIN_TASKS = ["task_easy"]  # 43-step AAPL uptrend — fast cache, clear reward signal

def log(msg: str) -> None:
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

log("Building precache plan for: " + str(TRAIN_TASKS))
council = Council(client=client, use_cache=True)

plan = []
for task_id in TRAIN_TASKS:
    env = StockerEnv(task_id=task_id)
    obs = env.reset().observation
    while True:
        for sp in council.specialists:
            cache_path = council._vote_cache_path(sp, obs)
            plan.append((sp, obs, cache_path.exists()))
        result = env.step({"side": "hold", "quantity": 0})
        if result.done:
            break
        obs = result.observation

todo  = [(sp, obs) for sp, obs, hit in plan if not hit]
hits  = sum(1 for _, _, hit in plan if hit)
total = len(plan)
log(f"Plan: {total} tuples — {hits} cached, {len(todo)} to compute.")
for sp_cls in council.specialists:
    name   = sp_cls.name
    n      = sum(1 for sp, _, _   in plan if sp.name == name)
    cached = sum(1 for sp, _, hit in plan if sp.name == name and hit)
    print(f"  {name:<18s} {cached:>3d}/{n:>3d} cached", flush=True)

if not todo:
    log("Nothing to do — all votes already cached.")
else:
    HEARTBEAT_EVERY = 25
    t0 = time.time()
    log(f"Computing {len(todo)} votes ... (heartbeat every {HEARTBEAT_EVERY})")
    pbar = tqdm(todo, desc="precaching", unit="vote", file=sys.stdout, dynamic_ncols=True)
    for i, (sp, obs) in enumerate(pbar, 1):
        council._cached_vote(sp, obs)
        pbar.set_postfix_str(f"{sp.name}@{obs.ticker}/{obs.date}")
        if i % HEARTBEAT_EVERY == 0 or i == len(todo):
            elapsed = time.time() - t0
            rate = i / max(elapsed, 1e-9)
            eta_s = (len(todo) - i) / max(rate, 1e-9)
            log(f"  {i}/{len(todo)} done  rate={rate:.2f} vote/s  ETA={eta_s/60:.1f} min")
    elapsed = time.time() - t0
    per_vote = elapsed / max(len(todo), 1)
    log(f"Done in {elapsed/60:.2f} min ({per_vote:.2f} s/vote). Cache: {total} entries.")


In [ ]:
# Pre-training baseline rollout (specialists from cache, moderator = base Gemma)
!python -m training.eval_rollout --tasks task_easy --out training/runs/eval_pre
!cat training/runs/eval_pre/summary.csv


In [ ]:
# GRPO on the moderator LoRA — task_easy only, T4-safe settings.
#
# Budget on free T4 (16 GB VRAM, 4-bit Gemma 4 E4B):
#   43 steps * 3 epochs / 4 batch = ~33 optimizer steps
#   ~45 s/step  →  ~25 min training
#
# TRL ≥0.13: (batch_size * grad_accum) % num_generations == 0
# Here: (4 * 1) % 4 == 0 ✓
!python -m training.train_grpo \
    --tasks task_easy \
    --epochs 3 \
    --num-generations 4 \
    --batch-size 4 \
    --grad-accum 1 \
    --lora-rank 16 \
    --lr 5e-6


In [ ]:
# Post-training eval — load the trained adapter into the same client.
import glob, os
RUN_DIR = sorted(glob.glob("training/runs/grpo_*"))[-1]
LORA_DIR = os.path.join(RUN_DIR, "moderator-lora")
print("Using LoRA:", LORA_DIR)

from peft import PeftModel
client.model = PeftModel.from_pretrained(client.model, LORA_DIR, adapter_name="moderator")
client.moderator_lora = "moderator"

!python -m training.eval_rollout --tasks task_easy --moderator-lora moderator --out training/runs/eval_post
!cat training/runs/eval_post/summary.csv


In [ ]:
# Show training + eval curves inline
from IPython.display import Image, display
import os
for p in [f"{RUN_DIR}/loss.png", f"{RUN_DIR}/reward.png",
          "training/runs/eval_pre/reward_curve.png",
          "training/runs/eval_post/reward_curve.png",
          "training/runs/eval_pre/portfolio_curve.png",
          "training/runs/eval_post/portfolio_curve.png"]:
    if os.path.exists(p):
        print(p)
        display(Image(p))

In [ ]:
# Compile all artifacts → training/runs/RESULTS.md
!python scripts/compile_results.py

# Push training artifacts to GitHub
import subprocess, os
result = subprocess.run(
    ["git", "add",
     "training/train_grpo.py",
     "training/train_grpo.ipynb",
     "training/runs/RESULTS.md",
     f"{RUN_DIR}/args.json",
     f"{RUN_DIR}/loss.png",
     f"{RUN_DIR}/reward.png",
    ],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
result2 = subprocess.run(
    ["git", "commit", "-m", f"Add GRPO training run {os.path.basename(RUN_DIR)} results"],
    capture_output=True, text=True
)
print(result2.stdout or result2.stderr)
result3 = subprocess.run(["git", "push"], capture_output=True, text=True)
print(result3.stdout or result3.stderr)
print("Done — artifacts pushed to GitHub.")
